# Transformar csv a parquet

In [2]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

archivo_csv = "../1_data_processed/v3_dataset_post_chi.csv"
archivo_parquet = "../1_data_processed/v3_dataset_post_chi.parquet"

os.makedirs("../1_data_processed/", exist_ok=True)

# Si ya existe, lo sobreescribimos
if os.path.exists(archivo_parquet):
    os.remove(archivo_parquet)

# Leer encabezado para definir tipos compactos
cols = pd.read_csv(archivo_csv, nrows=0).columns.tolist()

dtype_map = {}
for c in cols:
    if c.startswith("signo_zodiacal_"):
        dtype_map[c] = "uint8"
    elif c == "ESTANCIA_DIAS":
        dtype_map[c] = "int32"
    else:
        dtype_map[c] = "uint8"

chunksize = 100_000
writer = None
total = 0

for chunk in pd.read_csv(archivo_csv, dtype=dtype_map, chunksize=chunksize, low_memory=False):
    table = pa.Table.from_pandas(chunk, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(
            archivo_parquet,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)
    total += len(chunk)
    print(f"✅ Chunk convertido: {chunk.shape} | acumulado filas: {total:,}")

if writer is not None:
    writer.close()

print("✅ Parquet generado en:", archivo_parquet)

✅ Chunk convertido: (100000, 488) | acumulado filas: 100,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 200,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 300,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 400,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 500,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 600,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 700,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 800,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 900,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,000,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,100,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,200,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,300,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,400,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,500,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,600,000
✅ Chunk co

# Train/Test

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
CARPETA_SPLIT = Path("../1_data_processed/split_indices")
CARPETA_SPLIT.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.20
BASE_SEED = 42

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas target tipo 'signo_zodiacal_*'.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

label_multiclase = df[cols_signo].idxmax(axis=1)

from sklearn.model_selection import train_test_split

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=BASE_SEED,
    stratify=label_multiclase
)

np.save(CARPETA_SPLIT / "idx_train.npy", idx_train)
np.save(CARPETA_SPLIT / "idx_test.npy", idx_test)

print("✅ Split guardado")
print("Train:", len(idx_train))
print("Test:", len(idx_test))

Dataset cargado: (5808498, 488)
✅ Split guardado
Train: 4646798
Test: 1161700


# Modelado LR

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

USE_CV = True
N_SAMPLE = 100
N_ITER = 100
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_lr")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

LR_PARAMS = dict(
    solver="liblinear",
    C=1.0,
    max_iter=2000,
    random_state=BASE_SEED,
    class_weight="balanced"
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(**LR_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    if hasattr(modelo, "predict_proba"):
        try:
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
        except Exception:
            pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")
idx_test = np.load(RUTA_SPLIT / "idx_test.npy")

drop_cols = cols_signo.copy()
X_all = df.drop(columns=drop_cols).copy()

X_train_all = X_all.iloc[idx_train].reset_index(drop=True)
X_test_all = X_all.iloc[idx_test].reset_index(drop=True)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 LR -> {signo}")

    y_train_all = df_train[signo].astype(int)
    y_test_all = df_test[signo].astype(int)

    pos_idx = np.where(y_train_all.values == 1)[0]
    neg_idx = np.where(y_train_all.values == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all.iloc[sample_idx]
        y_boot = y_train_all.iloc[sample_idx]

        pos_rate = float(y_boot.mean())

        if USE_CV:
            fold_metrics = []
            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                X_tr, X_val = X_boot.iloc[tr_idx], X_boot.iloc[val_idx]
                y_tr, y_val = y_boot.iloc[tr_idx], y_boot.iloc[val_idx]

                m_cv = clone(MODELO)
                m_cv.fit(X_tr, y_tr)

                fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

            fold_metrics = np.array(fold_metrics, dtype=float)

            cv_accuracy = np.nanmean(fold_metrics[:, 0])
            cv_precision = np.nanmean(fold_metrics[:, 1])
            cv_recall = np.nanmean(fold_metrics[:, 2])
            cv_f1 = np.nanmean(fold_metrics[:, 3])
            cv_roc_auc = np.nanmean(fold_metrics[:, 4])
            cv_pr_auc = np.nanmean(fold_metrics[:, 5])
        else:
            cv_accuracy = np.nan
            cv_precision = np.nan
            cv_recall = np.nan
            cv_f1 = np.nan
            cv_roc_auc = np.nan
            cv_pr_auc = np.nan

        m_test = clone(MODELO)
        m_test.fit(X_boot, y_boot)
        test_acc, test_prec, test_rec, test_f1, test_roc, test_pr, test_tp, test_fp, test_tn, test_fn = evaluar_modelo(
            m_test, X_test_all, y_test_all
        )

        resultados.append({
            "modelo": "lr",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": pos_rate,
            "cv_accuracy": cv_accuracy,
            "cv_precision": cv_precision,
            "cv_recall": cv_recall,
            "cv_f1": cv_f1,
            "cv_roc_auc": cv_roc_auc,
            "cv_pr_auc": cv_pr_auc,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1,
            "test_roc_auc": test_roc,
            "test_pr_auc": test_pr,
            "test_tp": test_tp,
            "test_fp": test_fp,
            "test_tn": test_tn,
            "test_fn": test_fn
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

df_detalle = pd.DataFrame(resultados)
df_detalle.to_csv(CARPETA_OUT / "lr_detalle.csv", index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc", "test_pr_auc",
    "test_tp", "test_fp", "test_tn", "test_fn"
]

df_resumen = df_detalle.groupby("signo")[cols_metricas].agg(["mean", "std"])
df_resumen.to_csv(CARPETA_OUT / "lr_resumen.csv")

print("✅ LR terminado")

Dataset cargado: (5808498, 488)

🔮 LR -> signo_zodiacal_acuario


KeyboardInterrupt: 

# Modelado RF

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

USE_CV = True
N_SAMPLE = 100
N_ITER = 100
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_rf")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=2,
    random_state=BASE_SEED,
    class_weight="balanced"
)

def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        y_proba = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_proba)
        pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")
idx_test = np.load(RUTA_SPLIT / "idx_test.npy")

X_all = df.drop(columns=cols_signo).copy()
X_train_all = X_all.iloc[idx_train].reset_index(drop=True)
X_test_all = X_all.iloc[idx_test].reset_index(drop=True)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

resultados = []

for signo in cols_signo:
    print(f"\n🔮 RF -> {signo}")
    y_train_all = df_train[signo].astype(int)
    y_test_all = df_test[signo].astype(int)

    pos_idx = np.where(y_train_all.values == 1)[0]
    neg_idx = np.where(y_train_all.values == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it)
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all.iloc[sample_idx]
        y_boot = y_train_all.iloc[sample_idx]

        if USE_CV:
            fold_metrics = []
            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                m_cv = clone(MODELO)
                m_cv.fit(X_boot.iloc[tr_idx], y_boot.iloc[tr_idx])
                fold_metrics.append(evaluar_modelo(m_cv, X_boot.iloc[val_idx], y_boot.iloc[val_idx]))
            fold_metrics = np.array(fold_metrics, dtype=float)
            cv_accuracy = np.nanmean(fold_metrics[:, 0])
            cv_precision = np.nanmean(fold_metrics[:, 1])
            cv_recall = np.nanmean(fold_metrics[:, 2])
            cv_f1 = np.nanmean(fold_metrics[:, 3])
            cv_roc_auc = np.nanmean(fold_metrics[:, 4])
            cv_pr_auc = np.nanmean(fold_metrics[:, 5])
        else:
            cv_accuracy = cv_precision = cv_recall = cv_f1 = cv_roc_auc = cv_pr_auc = np.nan

        m_test = clone(MODELO)
        m_test.fit(X_boot, y_boot)
        test_acc, test_prec, test_rec, test_f1, test_roc, test_pr, test_tp, test_fp, test_tn, test_fn = evaluar_modelo(
            m_test, X_test_all, y_test_all
        )

        resultados.append({
            "modelo": "rf",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": cv_accuracy,
            "cv_precision": cv_precision,
            "cv_recall": cv_recall,
            "cv_f1": cv_f1,
            "cv_roc_auc": cv_roc_auc,
            "cv_pr_auc": cv_pr_auc,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1,
            "test_roc_auc": test_roc,
            "test_pr_auc": test_pr,
            "test_tp": test_tp,
            "test_fp": test_fp,
            "test_tn": test_tn,
            "test_fn": test_fn
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

df_detalle = pd.DataFrame(resultados)
df_detalle.to_csv(CARPETA_OUT / "rf_detalle.csv", index=False)
df_resumen = df_detalle.groupby("signo").agg(["mean", "std"])
df_resumen.to_csv(CARPETA_OUT / "rf_resumen.csv")
print("✅ RF terminado")

# Modelado DT

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

USE_CV = True
N_SAMPLE = 100
N_ITER = 100
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_dt")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=BASE_SEED,
    class_weight="balanced"
)

def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        y_proba = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_proba)
        pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")
idx_test = np.load(RUTA_SPLIT / "idx_test.npy")

X_all = df.drop(columns=cols_signo).copy()
X_train_all = X_all.iloc[idx_train].reset_index(drop=True)
X_test_all = X_all.iloc[idx_test].reset_index(drop=True)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

resultados = []

for signo in cols_signo:
    print(f"\n🔮 DT -> {signo}")
    y_train_all = df_train[signo].astype(int)
    y_test_all = df_test[signo].astype(int)

    pos_idx = np.where(y_train_all.values == 1)[0]
    neg_idx = np.where(y_train_all.values == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it)
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all.iloc[sample_idx]
        y_boot = y_train_all.iloc[sample_idx]

        if USE_CV:
            fold_metrics = []
            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                m_cv = clone(MODELO)
                m_cv.fit(X_boot.iloc[tr_idx], y_boot.iloc[tr_idx])
                fold_metrics.append(evaluar_modelo(m_cv, X_boot.iloc[val_idx], y_boot.iloc[val_idx]))
            fold_metrics = np.array(fold_metrics, dtype=float)
            cv_accuracy = np.nanmean(fold_metrics[:, 0])
            cv_precision = np.nanmean(fold_metrics[:, 1])
            cv_recall = np.nanmean(fold_metrics[:, 2])
            cv_f1 = np.nanmean(fold_metrics[:, 3])
            cv_roc_auc = np.nanmean(fold_metrics[:, 4])
            cv_pr_auc = np.nanmean(fold_metrics[:, 5])
        else:
            cv_accuracy = cv_precision = cv_recall = cv_f1 = cv_roc_auc = cv_pr_auc = np.nan

        m_test = clone(MODELO)
        m_test.fit(X_boot, y_boot)
        test_acc, test_prec, test_rec, test_f1, test_roc, test_pr, test_tp, test_fp, test_tn, test_fn = evaluar_modelo(
            m_test, X_test_all, y_test_all
        )

        resultados.append({
            "modelo": "dt",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": cv_accuracy,
            "cv_precision": cv_precision,
            "cv_recall": cv_recall,
            "cv_f1": cv_f1,
            "cv_roc_auc": cv_roc_auc,
            "cv_pr_auc": cv_pr_auc,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1,
            "test_roc_auc": test_roc,
            "test_pr_auc": test_pr,
            "test_tp": test_tp,
            "test_fp": test_fp,
            "test_tn": test_tn,
            "test_fn": test_fn
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

df_detalle = pd.DataFrame(resultados)
df_detalle.to_csv(CARPETA_OUT / "dt_detalle.csv", index=False)
df_resumen = df_detalle.groupby("signo").agg(["mean", "std"])
df_resumen.to_csv(CARPETA_OUT / "dt_resumen.csv")
print("✅ DT terminado")


🔮 DT -> signo_zodiacal_acuario
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_aries
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_capricornio


KeyboardInterrupt: 

# Modelado Knn

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

USE_CV = True
N_SAMPLE = 100
N_ITER = 100
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_knn")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=50,
        weights="uniform",
        metric="euclidean",
        n_jobs=1
    ))
])

def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        y_proba = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_proba)
        pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")
idx_test = np.load(RUTA_SPLIT / "idx_test.npy")

X_all = df.drop(columns=cols_signo).copy()
X_train_all = X_all.iloc[idx_train].reset_index(drop=True)
X_test_all = X_all.iloc[idx_test].reset_index(drop=True)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

resultados = []

for signo in cols_signo:
    print(f"\n🔮 KNN -> {signo}")
    y_train_all = df_train[signo].astype(int)
    y_test_all = df_test[signo].astype(int)

    pos_idx = np.where(y_train_all.values == 1)[0]
    neg_idx = np.where(y_train_all.values == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it)
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all.iloc[sample_idx]
        y_boot = y_train_all.iloc[sample_idx]

        if USE_CV:
            fold_metrics = []
            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                m_cv = clone(MODELO)
                m_cv.fit(X_boot.iloc[tr_idx], y_boot.iloc[tr_idx])
                fold_metrics.append(evaluar_modelo(m_cv, X_boot.iloc[val_idx], y_boot.iloc[val_idx]))
            fold_metrics = np.array(fold_metrics, dtype=float)
            cv_accuracy = np.nanmean(fold_metrics[:, 0])
            cv_precision = np.nanmean(fold_metrics[:, 1])
            cv_recall = np.nanmean(fold_metrics[:, 2])
            cv_f1 = np.nanmean(fold_metrics[:, 3])
            cv_roc_auc = np.nanmean(fold_metrics[:, 4])
            cv_pr_auc = np.nanmean(fold_metrics[:, 5])
        else:
            cv_accuracy = cv_precision = cv_recall = cv_f1 = cv_roc_auc = cv_pr_auc = np.nan

        m_test = clone(MODELO)
        m_test.fit(X_boot, y_boot)
        test_acc, test_prec, test_rec, test_f1, test_roc, test_pr, test_tp, test_fp, test_tn, test_fn = evaluar_modelo(
            m_test, X_test_all, y_test_all
        )

        resultados.append({
            "modelo": "knn",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": cv_accuracy,
            "cv_precision": cv_precision,
            "cv_recall": cv_recall,
            "cv_f1": cv_f1,
            "cv_roc_auc": cv_roc_auc,
            "cv_pr_auc": cv_pr_auc,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1,
            "test_roc_auc": test_roc,
            "test_pr_auc": test_pr,
            "test_tp": test_tp,
            "test_fp": test_fp,
            "test_tn": test_tn,
            "test_fn": test_fn
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

df_detalle = pd.DataFrame(resultados)
df_detalle.to_csv(CARPETA_OUT / "knn_detalle.csv", index=False)
df_resumen = df_detalle.groupby("signo").agg(["mean", "std"])
df_resumen.to_csv(CARPETA_OUT / "knn_resumen.csv")
print("✅ KNN terminado")

# Modelado SVM

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.svm import SVC

RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

USE_CV = True
N_SAMPLE = 100
N_ITER = 100
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_svm")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="linear",
        C=1.0,
        probability=True,
        random_state=BASE_SEED,
        class_weight="balanced"
    ))
])

def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        y_proba = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_proba)
        pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")
idx_test = np.load(RUTA_SPLIT / "idx_test.npy")

X_all = df.drop(columns=cols_signo).copy()
X_train_all = X_all.iloc[idx_train].reset_index(drop=True)
X_test_all = X_all.iloc[idx_test].reset_index(drop=True)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

resultados = []

for signo in cols_signo:
    print(f"\n🔮 SVM -> {signo}")
    y_train_all = df_train[signo].astype(int)
    y_test_all = df_test[signo].astype(int)

    pos_idx = np.where(y_train_all.values == 1)[0]
    neg_idx = np.where(y_train_all.values == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it)
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all.iloc[sample_idx]
        y_boot = y_train_all.iloc[sample_idx]

        if USE_CV:
            fold_metrics = []
            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                m_cv = clone(MODELO)
                m_cv.fit(X_boot.iloc[tr_idx], y_boot.iloc[tr_idx])
                fold_metrics.append(evaluar_modelo(m_cv, X_boot.iloc[val_idx], y_boot.iloc[val_idx]))
            fold_metrics = np.array(fold_metrics, dtype=float)
            cv_accuracy = np.nanmean(fold_metrics[:, 0])
            cv_precision = np.nanmean(fold_metrics[:, 1])
            cv_recall = np.nanmean(fold_metrics[:, 2])
            cv_f1 = np.nanmean(fold_metrics[:, 3])
            cv_roc_auc = np.nanmean(fold_metrics[:, 4])
            cv_pr_auc = np.nanmean(fold_metrics[:, 5])
        else:
            cv_accuracy = cv_precision = cv_recall = cv_f1 = cv_roc_auc = cv_pr_auc = np.nan

        m_test = clone(MODELO)
        m_test.fit(X_boot, y_boot)
        test_acc, test_prec, test_rec, test_f1, test_roc, test_pr, test_tp, test_fp, test_tn, test_fn = evaluar_modelo(
            m_test, X_test_all, y_test_all
        )

        resultados.append({
            "modelo": "svm",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": cv_accuracy,
            "cv_precision": cv_precision,
            "cv_recall": cv_recall,
            "cv_f1": cv_f1,
            "cv_roc_auc": cv_roc_auc,
            "cv_pr_auc": cv_pr_auc,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1,
            "test_roc_auc": test_roc,
            "test_pr_auc": test_pr,
            "test_tp": test_tp,
            "test_fp": test_fp,
            "test_tn": test_tn,
            "test_fn": test_fn
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

df_detalle = pd.DataFrame(resultados)
df_detalle.to_csv(CARPETA_OUT / "svm_detalle.csv", index=False)
df_resumen = df_detalle.groupby("signo").agg(["mean", "std"])
df_resumen.to_csv(CARPETA_OUT / "svm_resumen.csv")
print("✅ SVM terminado")

# Mostrar Resultados

In [1]:
import pandas as pd
from pathlib import Path

# =====================================================
# RUTAS DE LOS RESÚMENES
# =====================================================
BASE_PATH = Path("../5_results/bootstrap_cv_models")

resumenes = {
    "Random Forest": BASE_PATH / "rf_bootstrap_cv" / "rf_resumen.csv",
    "Árbol de Decisión": BASE_PATH / "dt_bootstrap_cv" / "dt_resumen.csv",
    "KNN": BASE_PATH / "knn_bootstrap_cv" / "knn_resumen.csv",
    "Regresión Logística": BASE_PATH / "lr_bootstrap_cv" / "lr_resumen.csv",
}

medias_modelos = {}

# =====================================================
# CARGA Y PROCESAMIENTO
# =====================================================
for nombre_modelo, ruta in resumenes.items():
    print(f"\n📊 {nombre_modelo}")
    print("-" * 50)

    df_resumen = pd.read_csv(ruta, header=[0, 1], index_col=0)

    # Extraer solo la media
    df_medias = df_resumen.xs("mean", axis=1, level=1)

    # Redondear
    df_medias = df_medias.round(3)
    

    medias_modelos[nombre_modelo] = df_medias

    display(df_medias.style.format("{:.3f}"))



📊 Random Forest
--------------------------------------------------


FileNotFoundError: [Errno 2] No such file or directory: '..\\5_results\\bootstrap_cv_models\\rf_bootstrap_cv\\rf_resumen.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =====================================================
# CONFIGURACIÓN
# =====================================================
BASE_PATH = Path("../5_results/bootstrap_cv_models")

MODELOS = {
    "Random Forest": BASE_PATH / "rf_bootstrap_cv" / "rf_detalle.csv",
    "Árbol de Decisión": BASE_PATH / "dt_bootstrap_cv" / "dt_detalle.csv",
    "KNN": BASE_PATH / "knn_bootstrap_cv" / "knn_detalle.csv",
    "Regresión Logística": BASE_PATH / "lr_bootstrap_cv" / "lr_detalle.csv",
}

signos = [
    'signo_zodiacal_acuario',
    'signo_zodiacal_aries',
    'signo_zodiacal_capricornio',
    'signo_zodiacal_cancer',
    'signo_zodiacal_escorpio',
    'signo_zodiacal_geminis',
    'signo_zodiacal_leo',
    'signo_zodiacal_libra',
    'signo_zodiacal_piscis',
    'signo_zodiacal_sagitario',
    'signo_zodiacal_tauro',
    'signo_zodiacal_virgo'
]

CARPETA_FIG = Path("../5_results/figures_bootstrap")
CARPETA_FIG.mkdir(parents=True, exist_ok=True)

# =====================================================
# FUNCIÓN DE GRÁFICO
# =====================================================
def plot_roc_auc_4x3(df, nombre_modelo, output_path):
    fig, axes = plt.subplots(4, 3, figsize=(16, 10), sharey=True)
    axes = axes.flatten()

    for ax, signo in zip(axes, signos):
        df_s = df[df["signo"] == signo].sort_values("iter")
        media = df_s["roc_auc"].mean()

        ax.scatter(df_s["iter"], df_s["roc_auc"], s=12, alpha=0.5)
        ax.axhline(0.5, color="red", linestyle="--", linewidth=1)
        ax.axhline(media, color="blue", linewidth=1.5)

        ax.set_title(
            f"{signo.replace('signo_zodiacal_', '').capitalize()} (μ = {media:.3f})",
            fontsize=11
        )

        ax.set_ylim(0.3, 0.7)
        ax.set_xlabel("Iteración")
        ax.set_ylabel("ROC-AUC")

    plt.suptitle(
        f"Distribución de ROC-AUC por iteración\n({nombre_modelo} + Bootstrap + CV)",
        fontsize=15
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(output_path, dpi=300)
    plt.close()

# =====================================================
# EJECUCIÓN POR MODELO
# =====================================================
for nombre_modelo, ruta_csv in MODELOS.items():
    print(f"📊 Generando figura para {nombre_modelo}")
    df = pd.read_csv(ruta_csv)

    out_img = CARPETA_FIG / f"roc_auc_4x3_{nombre_modelo.replace(' ', '_').lower()}.png"
    plot_roc_auc_4x3(df, nombre_modelo, out_img)

    print(f"   ✅ Guardado en: {out_img}")

print("\n✅ Todas las figuras generadas")


📊 Generando figura para Random Forest
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_random_forest.png
📊 Generando figura para Árbol de Decisión
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_árbol_de_decisión.png
📊 Generando figura para KNN
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_knn.png
📊 Generando figura para Regresión Logística
   ✅ Guardado en: ..\5_results\figures_bootstrap\roc_auc_4x3_regresión_logística.png

✅ Todas las figuras generadas
